# Saray kartı işaretleme · çıktı Drive'a kaydedilir (oturum kopsa da kalır)
Karakter verisi Saray İlişkileri Dosyası ile birebir. Her hesap kendi `saray_partN.jsonl`'ini etiketler.

**ÖNEMLİ (Colab):** Hem girdi hem çıktı **Drive'da** olmalı — `/content` oturum kopunca silinir.
1. `saray_partN.jsonl`'ini **Drive'a** koy: `MyDrive/kaydir_veri/` içine (ya da herhangi bir alt klasöre).
2. T4 GPU · 🔑 Secrets → `HF_TOKEN` · Tümünü çalıştır.
3. Çıktı Drive'a yazılır: `MyDrive/kaydir_veri/etiketleme/etiket_partN.jsonl`. Oturum kopsa bile durur, yeniden çalıştırınca **kaldığı yerden** devam eder.

**Kaggle:** GPU T4 x2 + Internet On + `HF_TOKEN` · dosyayı Add Input ile ekle · çıktı `/kaggle/working`'te, Output'tan indir.

## 1 · Ayar

In [ ]:
SURE_SAAT = 11   # en fazla kaç saat (Kaggle 12'de keser); 0 = sınırsız
GRUP = 8         # bir seferde kaç kart sorulsun

## 2 · Saray karakterleri — Saray İlişkileri Dosyası'ndan birebir

In [ ]:
SARAY = {
 "Vezir": {
  "deger": "Devletin düzeninin bugün değil, yıllar sonra da sağlam kalması.",
  "ister": "uzun vadeli düşünülmüş kararlar, beylerle uzlaşma, töreye ve kanuna uygunluk",
  "istemez": "aceleyle verilen hükümler, sonu düşünülmemiş savaşlar, devletin sözünü bozmak"
 },
 "Başkâtip": {
  "deger": "Her kararın kayda geçmesi ve devletin geçmişle tutarlı kalması.",
  "ister": "kararı eski kayıtlara dayandırmak, anlaşmaları yazılı yapmak, belgeleri ve defterleri korumak",
  "istemez": "kayıtsız ve belgesiz işler, eski fermanlarla çelişen kararlar, mühürsüz sözlü anlaşmalar"
 },
 "Hazinedar": {
  "deger": "Hazinenin dolu kalması ve her akçenin hesabının verilmesi.",
  "ister": "vergi gelirini koruyan kararlar, gereksiz harcamayı durdurmak, işe başlamadan hesabı görmek",
  "istemez": "hesapsız bağış ve şenlik, geliri kesen yasaklar, borçla yapılan işler"
 },
 "Kale Kumandanı": {
  "deger": "Kalenin sağlam, erzakın dolu ve askerin hazır olması.",
  "ister": "surları onarmak, erzak ambarlarını doldurmak, nöbeti ve disiplini sıkı tutmak",
  "istemez": "kaleyi boşaltıp akına çıkmak, askerin erzakından kesmek, kapıları tedbirsiz açmak"
 },
 "Akıncı Beyi": {
  "deger": "Ordunun cesur, hızlı ve ganimetle beslenen bir güç olması.",
  "ister": "akın ve keşif seferleri, atlara ve silaha harcama, düşmana hemen karşılık vermek",
  "istemez": "beklemek ve uzun pazarlık, ordudan para kesmek, düşmana haraç ödemek"
 },
 "Sınır Beyi": {
  "deger": "Sınır köylerinin ve geçitlerin güvende olması.",
  "ister": "geçitlere asker göndermek, sınır köylerini korumak, yabancı atlılara sert davranmak",
  "istemez": "sınırdan asker çekmek, komşuya toprak bırakmak, sınırı korumasız bırakan anlaşmalar"
 },
 "Elçi": {
  "deger": "Komşu krallıklarla barış ve verilen sözlerin tutulması.",
  "ister": "komşularla anlaşma yapmak, elçilere hediye göndermek, savaşı görüşmeyle önlemek",
  "istemez": "komşu topraklara akın, elçileri geri çevirmek, anlaşmayı bozmak"
 },
 "Saray Hatun": {
  "deger": "Hanedanın itibarı ve sarayın iç huzuru.",
  "ister": "hanedanın onurunu koruyan kararlar, komşularla incelikli uzlaşma, saray halkının huzuru",
  "istemez": "hanedanı küçük düşüren tavizler, sarayda kaba cezalar, düşünmeden savaşa girmek"
 },
 "Kervan Başı": {
  "deger": "Ticaret yollarının açık, güvenli ve kervanların kazançlı olması.",
  "ister": "yollara atlı devriye koymak, gümrüğü hafifletmek, hanları ve köprüleri onarmak",
  "istemez": "yolları kapatmak, kervanlardan ağır vergi almak, eşkıyaya göz yummak"
 },
 "Ahi Ustası": {
  "deger": "Çarşıda dürüst tartı, adil fiyat ve esnafın ekmeği.",
  "ister": "hileli esnafı cezalandırmak, yerli tezgâhları korumak, çırak yetiştirmek",
  "istemez": "çarşıya ağır vergi, ucuz yabancı malın serbestçe girmesi, hileye göz yummak"
 }
}

## 3 · Hazırlık: Drive'ı bağla, parçanı bul, HF
Çıktı Drive'a yazılır ki oturum kopunca kaybolmasın.

In [ ]:
import os, sys, subprocess, glob, json, time, getpass, hashlib
KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
if KAGGLE:
    aday = sorted(glob.glob('/kaggle/input/**/saray_part*.jsonl', recursive=True))
    CIKTI_KLASOR = '/kaggle/working'
else:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/kaydir_veri'
    CIKTI_KLASOR = f'{DRIVE}/etiketleme'; os.makedirs(CIKTI_KLASOR, exist_ok=True)
    aday = sorted(glob.glob(f'{DRIVE}/**/saray_part*.jsonl', recursive=True)
                  + glob.glob('/content/**/saray_part*.jsonl', recursive=True)
                  + glob.glob('saray_part*.jsonl'))
assert aday, "saray_partN.jsonl bulunamadı: Colab'da Drive'a (MyDrive/kaydir_veri/) koy; Kaggle'da Add Input ile ekle"
GIRDI = aday[0]
PARCA = ''.join(ch for ch in os.path.basename(GIRDI) if ch.isdigit()) or '0'
ETIKET = f'{CIKTI_KLASOR}/etiket_part{PARCA}.jsonl'; ARA = ETIKET + '.ara'
if not os.path.isdir('/tmp/AI'):
    subprocess.run(['git','clone','-q','--depth','1','https://github.com/IleriProgramlamaProje/AI','/tmp/AI'], check=True)
sys.path.insert(0, '/tmp/AI')
if 'motor' not in globals():
    subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','bitsandbytes'], check=True)
try:
    token = (__import__('kaggle_secrets').UserSecretsClient().get_secret('HF_TOKEN') if KAGGLE
             else __import__('google.colab', fromlist=['userdata']).userdata.get('HF_TOKEN'))
except Exception: token = None
from huggingface_hub import login
login(token=(token or getpass.getpass('Hugging Face token: ')).strip())
print(f'parça {PARCA} · girdi {GIRDI}')
print(f'çıktı → {ETIKET}  (oturum kopsa da burada kalır)')

## 4 · Kartları oku (bitenleri atla)

In [ ]:
from collections import Counter
kartlar = [json.loads(s) for s in open(GIRDI, encoding='utf-8') if s.strip()]
anahtar = lambda r: hashlib.md5(r['cikti'].encode('utf-8')).hexdigest()
bitti = {}
if os.path.exists(ARA):
    for s in open(ARA, encoding='utf-8'):
        d = json.loads(s); bitti[d['h']] = (d['c1'], d['c2'])
kalan = [{'karakter': r['karakter'], 'kart': r['cikti'], 'h': anahtar(r)} for r in kartlar if anahtar(r) not in bitti]
print(f'{len(kartlar)} kart · bitmiş {len(bitti)} · sorulacak {len(kalan)}')

## 5 · Modeli aç ve iki kez sor (çıktı her grupta Drive'a yazılır)

In [ ]:
import torch
from ai.veri_uretimi.uretici import motor_ac
if 'motor' not in globals():
    motor, _ = motor_ac('transformers8bit' if torch.cuda.device_count()>=2 else 'transformers4bit', 'google/gemma-4-12B-it')
SORU = '''Bir krallık oyununda bir kart var. Kartta {karakter} hükümdara bir dert anlatıyor ve iki karar sunuyor: Sol ve Sağ.
{karakter} en çok şuna değer verir: {deger}
{karakter} şunları İSTER: {ister}
{karakter} şunları İSTEMEZ: {istemez}

{kart}

Soru: {karakter} kendi değerlerine ve görüşüne göre hükümdarın hangi kararı vermesini ister?
Sadece tek kelime yaz: Sol, Sağ ya da Belirsiz.'''
def sor(grup):
    metinler=[motor.tok.apply_chat_template([{'role':'user','content':SORU.format(
        karakter=k['karakter'], kart=k['kart'], deger=SARAY[k['karakter']]['deger'],
        ister=SARAY[k['karakter']]['ister'], istemez=SARAY[k['karakter']]['istemez'])}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False) for k in grup]
    g = motor.tok(metinler, return_tensors='pt', padding=True, add_special_tokens=False).to(motor.model.device)
    with torch.no_grad():
        c = motor.model.generate(**g, do_sample=True, temperature=0.6, top_p=0.95, max_new_tokens=6, pad_token_id=motor.tok.pad_token_id)
    out=[]
    for satir in c[:, g['input_ids'].shape[1]:]:
        kel = motor.tok.decode(satir, skip_special_tokens=True).strip().split()
        ilk = (kel[0] if kel else '').strip('.:*,').lower()
        out.append('Sol' if ilk.startswith('sol') else 'Sağ' if ilk.startswith(('sağ','sag')) else 'Belirsiz')
    return out
t0=time.time()
with open(ARA,'a',encoding='utf-8') as f:
    for i in range(0,len(kalan),GRUP):
        if SURE_SAAT and time.time()-t0>SURE_SAAT*3600:
            print('Süre doldu, kaldığı yerde durdu. Yeniden çalıştır → devam.'); break
        grup=kalan[i:i+GRUP]
        for k,c1,c2 in zip(grup, sor(grup), sor(grup)):
            bitti[k['h']]=(c1,c2); f.write(json.dumps({'h':k['h'],'c1':c1,'c2':c2},ensure_ascii=False)+'\n')
        f.flush()
        try: os.fsync(f.fileno())
        except Exception: pass
        print(f'{min(i+GRUP,len(kalan))}/{len(kalan)}')

## 6 · Etiket dosyasını yaz (Drive'a)

In [ ]:
say=Counter()
with open(ETIKET,'w',encoding='utf-8') as f:
    for h,(c1,c2) in bitti.items():
        if c1==c2 and c1!='Belirsiz':
            f.write(json.dumps({'h':h,'istedigi':c1},ensure_ascii=False)+'\n'); say[c1]+=1
        else: say['belirsiz']+=1
print(f'parça {PARCA} bitti → {ETIKET}')
print(f'etiketlendi {say["Sol"]+say["Sağ"]} (Sol {say["Sol"]}, Sağ {say["Sağ"]}) · belirsiz {say["belirsiz"]}')
print('Drive\'da kaydedildi, oturum kopsa da durur. İndirmene gerek yok; birleştirmede Drive\'dan okunur.')

---
## BİRLEŞTİR  (5 parça bittikten sonra, 1 kez · GPU gerekmez)
5 etiket dosyası Drive'da `kaydir_veri/etiketleme/` içinde; `ham_birlesik.jsonl`'i de `kaydir_veri/` içine koy.

In [ ]:
import json, glob, hashlib, os
try:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE='/content/drive/MyDrive/kaydir_veri'
except Exception:
    DRIVE='.'
etiket={}
for yol in glob.glob(f'{DRIVE}/etiketleme/etiket_part*.jsonl') or glob.glob('etiket_part*.jsonl'):
    for s in open(yol,encoding='utf-8'):
        d=json.loads(s); etiket[d['h']]=d['istedigi']
GIR=f'{DRIVE}/ham_birlesik.jsonl'; CIK=f'{DRIVE}/ham_birlesik_istedigi.jsonl'
h=lambda r: hashlib.md5((r.get('cikti') or '').encode('utf-8')).hexdigest()
n=0
with open(CIK,'w',encoding='utf-8') as f:
    for s in open(GIR,encoding='utf-8'):
        if not s.strip(): continue
        r=json.loads(s)
        if r.get('tur')=='kart' and h(r) in etiket: r['istedigi']=etiket[h(r)]; n+=1
        f.write(json.dumps(r,ensure_ascii=False)+'\n')
print(f'{len(etiket)} etiket · {n} karta işlendi → {CIK}')